# GIAI ĐOẠN 2: TIỀN HUẤN LUYỆN BASE MODEL (PRE-TRAINING) & PROFILING KV CACHE

> **Dự án:** Xây dựng Large Language Model from Scratch (Clean Architecture)  
> **Phần cứng mục tiêu:** Kaggle GPU (T4 16GB VRAM) / Colab GPU  
> **Các tính năng bắt buộc tích hợp:**  
> - **Weights & Biases (W&B):** Tự động theo dõi train/val loss, perplexity, learning rate, VRAM và text samples.  
> - **Tự động Resume Training:** Nếu phiên bị ngắt kết nối giữa chừng, tự động nạp `last_checkpoint.pt` để train tiếp mà không phải train lại từ đầu.  
> - **Kỹ thuật tối ưu VRAM:** Mixed Precision (`torch.amp.autocast`), Gradient Accumulation, Gradient Clipping.  
> - **Đo đạc định lượng (Profiling) KV Cache:** Tốc độ (Tokens/s, TTFT, TPOT) và Bộ nhớ (Peak VRAM) khi **Bật KV Cache vs Tắt KV Cache** tại 4 mốc ngữ cảnh $L \in \{128, 512, 1024, 2048\}$ kèm bảng tổng hợp và biểu đồ trực quan.

## 1. Thiết lập Môi trường & Khởi tạo Weights & Biases (W&B)

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q wandb datasets matplotlib
!nvidia-smi

import os
import sys

# Đảm bảo import được module 'src' từ thư mục gốc của repo
project_root = os.path.abspath(".")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
import wandb
from src.utils.tracker import WandbTracker

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

# Đăng nhập W&B (Điền API Key của bạn hoặc dùng chế độ 'offline' / 'disabled' nếu muốn chạy ẩn)
# wandb.login(key="YOUR_WANDB_API_KEY")
USE_WANDB = True
WANDB_MODE = "online"  # Chọn: "online", "offline", hoặc "disabled"

## 2. Chuẩn bị Dữ liệu Huấn luyện (TinyStories Dataset)
Tải tập TinyStories và mã hóa thành binary `train.bin`, `val.bin` kiểu `uint16` để nạp siêu tốc qua `numpy.memmap`.

In [ ]:
import numpy as np
from datasets import load_dataset
from src.tokenizer import BPETokenizer

tokenizer = BPETokenizer()
vocab_file = "checkpoints/vocab.json"
merges_file = "checkpoints/merges.txt"

if os.path.exists(vocab_file) and os.path.exists(merges_file):
    print("Nạp từ điển Tokenizer có sẵn...")
    tokenizer.load_vocab(vocab_file, merges_file)
else:
    print("Khởi tạo từ điển Tokenizer mặc định...")

os.makedirs("data", exist_ok=True)
train_bin_path = "data/train.bin"
val_bin_path = "data/val.bin"

if not os.path.exists(train_bin_path):
    print("📥 Đang tải dataset TinyStories từ HuggingFace...")
    ds = load_dataset("roneneldan/TinyStories", split="train[:50000]")
    print(f"Số lượng mẫu: {len(ds):,}")

    print("⚡ Đang Token hóa dữ liệu...")
    all_tokens = []
    for sample in ds:
        tokens = tokenizer.encode(sample["text"]) + [tokenizer.special_tokens.get("<|eos|>", 2)]
        all_tokens.extend(tokens)

    all_tokens_np = np.array(all_tokens, dtype=np.uint16)
    split_idx = int(len(all_tokens_np) * 0.95)
    all_tokens_np[:split_idx].tofile(train_bin_path)
    all_tokens_np[split_idx:].tofile(val_bin_path)
    print(f"✅ Đã lưu {train_bin_path} ({len(all_tokens_np[:split_idx]):,} tokens)")
    print(f"✅ Đã lưu {val_bin_path} ({len(all_tokens_np[split_idx:]):,} tokens)")
else:
    print("✅ Dữ liệu train.bin đã tồn tại sẵn, bỏ qua bước tokenize.")

## 3. Khởi tạo Mô hình & Cấu hình Siêu tham số (Fit 16GB VRAM)

In [ ]:
from src.config import ModelConfig
from src.model import GPTModel

device = "cuda" if torch.cuda.is_available() else "cpu"

# Cấu hình siêu tham số mô hình fit 16GB VRAM: ~45M tham số
model_cfg = ModelConfig(
    vocab_size=8192,
    block_size=512,       # Context window L=512 khi pre-train
    n_layers=8,
    n_heads=8,
    d_model=512,
    tie_weights=True,
)

model = GPTModel(model_cfg).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"🚀 GPTModel đã khởi tạo trên {device}!")
print(f"Tổng số tham số: {total_params / 1e6:.2f}M (Vừa vặn GPU 16GB)")

## 4. Vòng lặp Huấn luyện (AMP, Grad Accum, Cosine LR, W&B Tracking, Auto-Resume)

> **Tính năng Auto-Resume:** Nếu quá trình huấn luyện bị ngắt quãng giữa chừng, bạn chỉ cần bấm chạy lại cell này! CheckpointManager sẽ tự động nạp `checkpoints/last_checkpoint.pt`, khôi phục step, epoch, optimizer, scheduler và log nối tiếp lên W&B mà không phải train từ đầu.

In [ ]:
import math
import time
import torch.nn as nn
from src.utils.checkpoint import CheckpointManager

# Siêu tham số huấn luyện
BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 4           # Effective batch size = 32 * 4 = 128 chuỗi
MAX_LR = 5e-4
MIN_LR = 5e-5
WARMUP_STEPS = 300
MAX_STEPS = 5000
SAVE_CHECKPOINT_EVERY = 500   # Lưu checkpoint định kỳ mỗi 500 steps
CHECKPOINT_DIR = "checkpoints"
LAST_CKPT_PATH = os.path.join(CHECKPOINT_DIR, "last_checkpoint.pt")
BEST_CKPT_PATH = os.path.join(CHECKPOINT_DIR, "base_model.pt")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Khởi tạo Optimizer & Mixed Precision Scaler
optimizer = torch.optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=0.1, betas=(0.9, 0.95))
use_amp = (device == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

# Tự động Resume từ checkpoint trước đó nếu có
start_step = 0
best_val_loss = float("inf")

if os.path.exists(LAST_CKPT_PATH):
    print(f"🔄 Đang phục hồi trạng thái huấn luyện từ: {LAST_CKPT_PATH}...")
    meta = CheckpointManager.load_checkpoint(
        LAST_CKPT_PATH,
        model=model,
        optimizer=optimizer,
        scaler=scaler if use_amp else None,
        device=device,
    )
    start_step = meta.get("step", 0)
    best_val_loss = meta.get("best_val_loss", float("inf"))
    print(f"✅ Tiếp tục huấn luyện từ Step {start_step:,} | Best Val Loss trước đó: {best_val_loss:.4f}")
else:
    print("🌱 Bắt đầu huấn luyện mới từ Step 0.")

# Khởi tạo Weights & Biases Tracker
tracker = WandbTracker(
    project_name="LLM-DecoderOnly-Scratch",
    run_name="Pretrain-BaseModel-TinyStories",
    config={
        "vocab_size": model_cfg.vocab_size,
        "d_model": model_cfg.d_model,
        "n_layers": model_cfg.n_layers,
        "n_heads": model_cfg.n_heads,
        "batch_size": BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "max_lr": MAX_LR,
        "warmup_steps": WARMUP_STEPS,
        "max_steps": MAX_STEPS,
    },
    enabled=USE_WANDB,
    mode=WANDB_MODE,
)

# Đọc dữ liệu qua memmap
train_data = np.memmap("data/train.bin", dtype=np.uint16, mode="r")
val_data = np.memmap("data/val.bin", dtype=np.uint16, mode="r")

def get_batch(split="train"):
    data = train_data if split == "train" else val_data
    ix = np.random.randint(0, len(data) - model_cfg.block_size, size=BATCH_SIZE)
    x = torch.stack([torch.from_numpy(data[i : i + model_cfg.block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1 : i + 1 + model_cfg.block_size].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

def get_lr(it):
    if it < WARMUP_STEPS:
        return MAX_LR * (it + 1) / WARMUP_STEPS
    if it > MAX_STEPS:
        return MIN_LR
    decay_ratio = (it - WARMUP_STEPS) / (MAX_STEPS - WARMUP_STEPS)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return MIN_LR + coeff * (MAX_LR - MIN_LR)

@torch.no_grad()
def estimate_loss(eval_iters=50):
    model.eval()
    losses = {}
    criterion = nn.CrossEntropyLoss()
    for split in ["train", "val"]:
        loss_sum = 0.0
        for _ in range(eval_iters):
            x, y = get_batch(split)
            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                logits = model(x)
                loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            loss_sum += loss.item()
        losses[split] = loss_sum / eval_iters
    model.train()
    return losses

In [ ]:
# VÒNG LẶP HUẤN LUYỆN CHÍNH (TRAINING LOOP)
criterion = nn.CrossEntropyLoss()
model.train()
t0 = time.time()

for step in range(start_step, MAX_STEPS):
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    optimizer.zero_grad()
    for micro_step in range(GRAD_ACCUM_STEPS):
        x, y = get_batch("train")
        with torch.amp.autocast(device_type="cuda", enabled=use_amp):
            logits = model(x)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            loss = loss / GRAD_ACCUM_STEPS

        if use_amp:
            scaler.scale(loss).backward()
        else:
            loss.backward()

    if use_amp:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
    else:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    # Log tiến trình và gửi metrics lên W&B
    if (step + 1) % 50 == 0 or step == start_step:
        dt = time.time() - t0
        t0 = time.time()
        current_loss = loss.item() * GRAD_ACCUM_STEPS
        current_vram = torch.cuda.max_memory_allocated() / (1024**2) if use_amp else 0.0
        
        print(f"Step {step + 1:5d}/{MAX_STEPS} | Loss: {current_loss:.4f} | LR: {lr:.6f} | VRAM: {current_vram:.1f}MB | {dt:.2f}s/50steps")
        
        tracker.log({
            "train/loss": current_loss,
            "train/learning_rate": lr,
            "system/peak_vram_mb": current_vram,
        }, step=step + 1)

    # Đánh giá Validation và Lưu Checkpoint
    if (step + 1) % SAVE_CHECKPOINT_EVERY == 0 or (step + 1) == MAX_STEPS:
        losses = estimate_loss()
        val_ppl = math.exp(min(losses["val"], 20))
        print(f"\n>>> [ĐÁNH GIÁ] Step {step + 1}: Train Loss = {losses['train']:.4f}, Val Loss = {losses['val']:.4f}, Val Perplexity = {val_ppl:.2f}")

        # Gửi validation metrics lên W&B
        tracker.log({
            "val/loss": losses["val"],
            "val/perplexity": val_ppl,
        }, step=step + 1)

        # 1. Luôn lưu last_checkpoint.pt để sẵn sàng Resume
        CheckpointManager.save_checkpoint(
            LAST_CKPT_PATH,
            model=model,
            optimizer=optimizer,
            scaler=scaler if use_amp else None,
            step=step + 1,
            best_val_loss=best_val_loss,
            stage="pretrain",
        )
        print(f"💾 Đã lưu checkpoint phục hồi tại: {LAST_CKPT_PATH}")

        # 2. Lưu base_model.pt khi đạt kỷ lục mới
        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            CheckpointManager.save_base_model(model, BEST_CKPT_PATH)
            print(f"🌟 Đã lưu kỷ lục Base Model tốt nhất tại: {BEST_CKPT_PATH}")
        print("-" * 70)

## 5. Đo đạc Định lượng (Profiling): Bật KV Cache vs. Tắt KV Cache

Thực hiện đo định lượng chính xác hiệu năng tại 4 mốc độ dài ngữ cảnh:  
$$L \in \{128, 512, 1024, 2048\}$$

Ghi nhận đủ 4 chỉ số theo quy định Checkpoint 2:
1. **Tốc độ sinh từ (`Tokens/giây`)** — Càng cao càng tốt.
2. **First Token Latency (TTFT)** — Thời gian xử lý prompt ban đầu và sinh token đầu tiên (ms).
3. **Next Token Latency (TPOT)** — Thời gian trung bình để giải mã từng token tiếp theo (ms).
4. **Mức chiếm dụng bộ nhớ đỉnh (Peak VRAM)** — Đo trực tiếp qua `torch.cuda.max_memory_allocated()`.

In [ ]:
from src.utils.profiler import KVCacheProfiler
import pandas as pd
import json

# Nạp trọng số mô hình tốt nhất vừa huấn luyện
if os.path.exists(BEST_CKPT_PATH):
    CheckpointManager.load_base_model(model, BEST_CKPT_PATH, device=device)
model.eval()

context_lengths = [128, 512, 1024, 2048]
benchmark_results = []

print("=" * 80)
print("📊 TIẾN HÀNH ĐO ĐẠC ĐỊNH LƯỢNG: CÓ KV CACHE VS. KHÔNG CÓ KV CACHE")
print("=" * 80)

for L in context_lengths:
    print(f"\n>>> Đang đo mốc Context Length L = {L} (Sinh 64 tokens)...")
    
    # 1. Đo khi BẬT KV Cache
    res_on = KVCacheProfiler.measure(model, prompt_length=L, generate_tokens=64, use_kv_cache=True, device=device)
    print(f"  [BẬT KV CACHE]  -> Tốc độ: {res_on.tokens_per_sec:6.2f} tok/s | TTFT: {res_on.ttft_ms:6.2f} ms | TPOT: {res_on.tpot_ms:6.2f} ms | VRAM: {res_on.peak_vram_mb:6.2f} MB")
    benchmark_results.append(vars(res_on))

    # 2. Đo khi TẮT KV Cache
    res_off = KVCacheProfiler.measure(model, prompt_length=L, generate_tokens=64, use_kv_cache=False, device=device)
    print(f"  [TẮT KV CACHE]  -> Tốc độ: {res_off.tokens_per_sec:6.2f} tok/s | TTFT: {res_off.ttft_ms:6.2f} ms | TPOT: {res_off.tpot_ms:6.2f} ms | VRAM: {res_off.peak_vram_mb:6.2f} MB")
    benchmark_results.append(vars(res_off))

# Lưu kết quả đo định lượng dạng JSON
os.makedirs("docs/figures", exist_ok=True)
with open("docs/figures/kv_cache_metrics.json", "w", encoding="utf-8") as f:
    json.dump(benchmark_results, f, indent=4)

# Hiển thị bảng tổng hợp so sánh trực quan (Pandas DataFrame)
table_rows = []
for L in context_lengths:
    on = next(r for r in benchmark_results if r["context_length"] == L and r["use_kv_cache"])
    off = next(r for r in benchmark_results if r["context_length"] == L and not r["use_kv_cache"])
    speedup = on["tokens_per_sec"] / max(off["tokens_per_sec"], 1e-6)
    table_rows.append({
        "Context (L)": L,
        "KV Cache": "BẬT",
        "Speed (tok/s)": on["tokens_per_sec"],
        "TTFT (ms)": on["ttft_ms"],
        "TPOT (ms)": on["tpot_ms"],
        "Peak VRAM (MB)": on["peak_vram_mb"],
        "Speedup": f"{speedup:.2f}x",
    })
    table_rows.append({
        "Context (L)": L,
        "KV Cache": "TẮT",
        "Speed (tok/s)": off["tokens_per_sec"],
        "TTFT (ms)": off["ttft_ms"],
        "TPOT (ms)": off["tpot_ms"],
        "Peak VRAM (MB)": off["peak_vram_mb"],
        "Speedup": "1.00x",
    })

df_metrics = pd.DataFrame(table_rows)
display(df_metrics)

# Đẩy bảng metrics lên Weights & Biases
tracker.log_text_samples("benchmark/kv_cache_comparison", table_rows)

## 6. Xuất Biểu đồ So sánh Hiệu năng (Checkpoint 2 Artifact)
Vẽ 4 biểu đồ trực quan đối chiếu Tốc độ, Độ trễ (TTFT, TPOT) và Bộ nhớ (Peak VRAM) giữa Bật vs Tắt KV Cache.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

metrics = [
    ("tokens_per_sec", "Tốc độ sinh (Tokens/s) - [Càng cao càng tốt]", "Tokens/giây"),
    ("ttft_ms", "First Token Latency - TTFT (ms) - [Càng thấp càng tốt]", "Mili-giây (ms)"),
    ("tpot_ms", "Next Token Latency - TPOT (ms) - [Càng thấp càng tốt]", "Mili-giây (ms)"),
    ("peak_vram_mb", "Chiếm dụng Peak VRAM (MB)", "Megabytes (MB)"),
]

for idx, (m_key, title, ylabel) in enumerate(metrics):
    ax = axes[idx]
    with_kv = [next(r[m_key] for r in benchmark_results if r["context_length"] == L and r["use_kv_cache"]) for L in context_lengths]
    without_kv = [next(r[m_key] for r in benchmark_results if r["context_length"] == L and not r["use_kv_cache"]) for L in context_lengths]

    x_indices = np.arange(len(context_lengths))
    width = 0.35

    ax.bar(x_indices - width/2, with_kv, width=width, label="Bật KV Cache", color="#2ca02c")
    ax.bar(x_indices + width/2, without_kv, width=width, label="Tắt KV Cache", color="#d62728")

    ax.set_xticks(x_indices)
    ax.set_xticklabels([f"L={L}" for L in context_lengths], fontweight="bold")
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.7)

plt.suptitle("KẾT QUẢ ĐO ĐẠC THỰC NGHIỆM ĐỊNH LƯỢNG HIỆU NĂNG KV CACHE", fontsize=14, fontweight="bold")
plt.tight_layout()
plot_path = "docs/figures/kv_cache_benchmark.png"
plt.savefig(plot_path, dpi=300)
plt.show()
print(f"📊 Đã xuất và lưu biểu đồ tại: {plot_path}")

# Log biểu đồ lên W&B nếu đang bật
if tracker.enabled and tracker.run:
    wandb.log({"benchmark/chart": wandb.Image(plot_path)})

## 7. Demo Sinh Văn Bản Tự Hồi Quy & Đóng W&B Session

In [ ]:
prompts = [
    "Once upon a time, there was a little girl named Lily",
    "One sunny day, a little dog found a magic ball",
    "In a small village near the mountains",
]

print("=" * 75)
print("✨ DEMO SINH VĂN BẢN VỚI BASE MODEL (KV CACHE ENABLED)")
print("=" * 75)

demo_samples = []
for p in prompts:
    input_ids = torch.tensor([tokenizer.encode(p)], device=device)
    output_ids = model.generate(
        input_ids,
        max_new_tokens=50,
        temperature=0.8,
        top_k=40,
        top_p=0.9,
        use_kv_cache=True,
    )
    gen_text = tokenizer.decode(output_ids[0].tolist())
    print(f"\n[Prompt]: {p}")
    print(f"[Generated]:\n{gen_text}")
    print("-" * 50)
    demo_samples.append({"prompt": p, "generated_text": gen_text})

# Log các mẫu sinh văn bản lên W&B
tracker.log_text_samples("demo/generations", demo_samples)

# Kết thúc theo dõi W&B
tracker.finish()
print("🎉 Toàn bộ quá trình Huấn luyện & Profiling đã hoàn tất thành công!")